# P1–P5 flagship evidence

## Scope and authority

The five flagship identities are P1 Recursive Epistemic Reconstruction, P2 Open-World Scientific Knowledge Discovery, P3 Global Knowledge Portrait, P4 Verified Scientific Discovery, and P5 Self-ORION. This notebook displays their receipt-derived measurements without treating unlike metrics as commensurate. All five stronger external gates remain bounded by their recorded authority state.


## Theory, methodology, and algorithms

A discovery ranking metric may be displayed with the standard definition

$$\operatorname{DCG}@k=\sum_{i=1}^{k}\frac{2^{\mathrm{rel}_i}-1}{\log_2(i+1)},\qquad
\operatorname{nDCG}@k=\frac{\operatorname{DCG}@k}{\operatorname{IDCG}@k}.$$

But a programme terminal is a **vector gate**, not a single favorable coordinate:

$$G=\bigwedge_{j=1}^{d} g_j(m_j,\tau_j).$$

Therefore a favorable nDCG coordinate does not override P2's recorded overall `FAIL`. For recursive reconstruction, a schematic update is $S_{t+1}=\mathcal{R}(S_t,e_t)$; for claim admission, evidence identity, execution identity, and authority remain separate inputs. These equations explain the reading logic; they are not new empirical results.


In [ ]:
from pathlib import Path
import json
import sys


def find_visualization_root(start=Path.cwd()):
    """Find visualization/ whether Jupyter starts at the repo root or notebooks/."""
    start = start.resolve()
    candidates = [start / "visualization", start, *start.parents]
    for candidate in candidates:
        if candidate.name == "visualization" and (candidate / "data" / "derived" / "atlas.json").exists():
            return candidate
        nested = candidate / "visualization"
        if (nested / "data" / "derived" / "atlas.json").exists():
            return nested
    raise FileNotFoundError(
        "Could not find visualization/data/derived/atlas.json. "
        "Build the atlas from the repository root first."
    )


VIS_ROOT = find_visualization_root()
sys.path.insert(0, str(VIS_ROOT / "src"))
ATLAS_PATH = VIS_ROOT / "data" / "derived" / "atlas.json"
atlas = json.loads(ATLAS_PATH.read_text(encoding="utf-8"))


def as_rows(value):
    """Return normalized records without changing their scientific values."""
    if isinstance(value, list):
        return [row for row in value if isinstance(row, dict)]
    if isinstance(value, dict):
        return [row for row in value.values() if isinstance(row, dict)]
    return []


def first(row, *keys, default=None):
    for key in keys:
        if key in row and row[key] is not None:
            return row[key]
    return default


def paper_id(row):
    raw = str(first(row, "paper_id", "paper", "id", default="UNSCOPED"))
    return raw.replace("ORION-", "")


def exact_status(row):
    return str(first(row, "terminal", "status", "result_state", "authority", default="UNSPECIFIED"))


def numeric_value(row):
    value = first(row, "value", "observed", "count", default=None)
    return float(value) if isinstance(value, (int, float)) and not isinstance(value, bool) else None


paper_states = as_rows(atlas.get("paper_states", []))
metrics_by_paper = atlas.get("metrics", {})
metrics = as_rows(atlas.get("metric_records", []))
anomalies = as_rows(atlas.get("anomalies", []))
sources = as_rows(atlas.get("sources", []))
des_execution = as_rows(atlas.get("des_execution", []))
framework_mechanics = atlas.get("framework_mechanics", {})

print(f"Atlas: {ATLAS_PATH}")
print(
    f"Loaded {len(paper_states)} paper states, {len(metrics)} metrics, "
    f"{len(anomalies)} anomalies, {len(des_execution)} frozen DES rows and "
    f"{len(sources)} sources."
)


In [ ]:
import matplotlib.pyplot as plt
import textwrap
from matplotlib.colors import ListedColormap  # noqa: F401 -- used by heatmap notebooks

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 10,
})

STATE_COLORS = {
    "PASS": "#2e7d32",
    "SUPPORTED": "#2e7d32",
    "FAIL": "#c62828",
    "GATE_NOT_MET": "#c62828",
    "CANNOT_CHECK": "#ef6c00",
    "UNKNOWN": "#6a1b9a",
    "NOT_AUTHORITY": "#455a64",
    "NOT_EXECUTED": "#757575",
}


def state_color(text):
    upper = str(text).upper()
    for token, color in STATE_COLORS.items():
        if token in upper:
            return color
    return "#1565c0"


def human_label(value, width=18):
    # Wrap machine identifiers without changing canonical capitalization.
    cleaned = str(value).replace("_", " ").replace(":", " — ")
    return "\n".join(textwrap.wrap(cleaned, width=width, break_long_words=False))



def print_records(rows, fields, limit=30):
    """Small dependency-free table for exact atlas fields."""
    rows = list(rows)
    if not rows:
        print("No records match the current display selectors.")
        return
    widths = {
        field: min(
            48,
            max(len(field), *(len(str(first(row, field, default=""))) for row in rows[:limit])),
        )
        for field in fields
    }
    print(" | ".join(field.ljust(widths[field]) for field in fields))
    print("-+-".join("-" * widths[field] for field in fields))
    for row in rows[:limit]:
        print(" | ".join(str(first(row, field, default=""))[: widths[field]].ljust(widths[field]) for field in fields))
    if len(rows) > limit:
        print(f"... {len(rows) - limit} more record(s); change DISPLAY_LIMIT to inspect them.")


## Editable selectors and display thresholds

Choose one exact metric name at a time so units and estimands are not mixed. `MIN_VALUE` is only a display filter and defaults to no filtering.


In [ ]:
PAPERS = ["P1", "P2", "P3", "P4", "P5"]
available_metric_names = sorted({str(first(row, "metric", "name", "metric_name", default="")) for row in metrics if paper_id(row) in PAPERS and numeric_value(row) is not None})
METRIC_NAME = "recall_at_100" if "recall_at_100" in available_metric_names else (available_metric_names[0] if available_metric_names else None)
MIN_VALUE = None
DISPLAY_LIMIT = 40
print("Available exact metric names:", available_metric_names)
print("Selected:", METRIC_NAME)


## Results: raw values and one-metric ECDF

Every point is an atlas record, directly labelled by paper and method. Rate metrics use their full 0–1 domain so small differences are not magnified by a cropped axis. The right panel is the empirical cumulative distribution of the same selected metric; it does not smooth a small sample or mix metrics.


In [ ]:
selected_metrics = [
    row for row in metrics
    if paper_id(row) in PAPERS
    and numeric_value(row) is not None
    and (METRIC_NAME is None or str(first(row, "metric", "name", "metric_name", default="")) == METRIC_NAME)
    and (MIN_VALUE is None or numeric_value(row) >= MIN_VALUE)
]
fig, (ax_values, ax_ecdf) = plt.subplots(1, 2, figsize=(13, 5.5), gridspec_kw={"width_ratios": [1.25, 1]})
if selected_metrics:
    rows = sorted(
        selected_metrics,
        key=lambda row: (paper_id(row), str(first(row, "name", "arm", "case_id", default=""))),
    )
    values = [numeric_value(row) for row in rows]
    display_names = {
        "bm25": "BM25",
        "orion_full": "ORION full",
        "orion_strong_new": "ORION strong-new",
        "rrf_hybrid": "RRF hybrid",
    }
    labels = [
        f"{paper_id(row)} — {human_label(display_names.get(str(first(row, 'name', 'arm', 'case_id', default='receipt')), first(row, 'name', 'arm', 'case_id', default='receipt')), 24)}"
        for row in rows
    ]
    unit = str(first(rows[0], "unit", default="receipt unit"))
    metric_label = human_label(METRIC_NAME, 32)
    y = list(range(len(rows)))
    ax_values.scatter(values, y, s=52, color="#1565c0", zorder=3)
    ax_values.set_yticks(y, labels)
    ax_values.invert_yaxis()
    ax_values.set_title(f"Raw receipt values: {metric_label}")
    ax_values.set_xlabel(f"{metric_label} ({unit})")
    ax_values.set_ylabel("paper and method")
    for value, ypos in zip(values, y):
        ax_values.annotate(
            f"{value:.3f}",
            (value, ypos),
            xytext=(6, 0),
            textcoords="offset points",
            va="center",
            fontsize=9,
        )
    ordered = sorted(values)
    cumulative = [(index + 1) / len(ordered) for index in range(len(ordered))]
    ax_ecdf.step(ordered, cumulative, where="post", color="#1565c0", linewidth=2)
    ax_ecdf.scatter(ordered, cumulative, color="#1565c0", s=28, zorder=3)
    ax_ecdf.set_title(f"Empirical cumulative distribution (n={len(ordered)})")
    ax_ecdf.set_xlabel(f"{metric_label} ({unit})")
    ax_ecdf.set_ylabel("cumulative fraction")
    ax_ecdf.set_ylim(0, 1.02)
    if unit.lower() in {"rate", "fraction", "proportion", "probability"}:
        for ax in (ax_values, ax_ecdf):
            ax.set_xlim(0, 1)
    fig.text(
        0.5,
        0.01,
        "Receipt-level display only; no favorable direction or external authority is inferred.",
        ha="center",
        fontsize=9,
        color="#455a64",
    )
else:
    for ax in (ax_values, ax_ecdf):
        ax.text(0.5, 0.5, "No matching numeric records", ha="center", va="center")
        ax.set_axis_off()
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()


In [ ]:
print_records(selected_metrics, ["paper_id", "metric", "name", "value", "unit", "status", "source_id"], DISPLAY_LIMIT)

flagship_anomalies = [row for row in anomalies if paper_id(row) in PAPERS]
print(f"\nP1–P5 anomaly records (unfiltered): {len(flagship_anomalies)}")
print_records(flagship_anomalies, ["paper_id", "anomaly_id", "severity", "status", "summary"], DISPLAY_LIMIT)


## Discussion: load-bearing anomalies

- **P2:** the overall result remains `FAIL` even when an nDCG coordinate looks favorable. A single metric must not replace the frozen multi-endpoint terminal.
- **P5:** the requested model was `glm-5.2`, while the served model was `glm-5.3`. This is an execution-identity mismatch, not evidence for the requested-model condition.

Inspect the linked receipts to explain mechanism; the visualization intentionally does not guess missing causes.

## Claim ceiling

These plots show exact normalized receipt values at their registered scope. They do not prove that the flagship algorithms generalize, outperform alternatives, or satisfy independent/top-tier publication gates.
